# Calibration Notebook - Camera Calibration as Correspondences

**Applied Computer Vision, TU Berlin - 2026**

In this notebook you will treat camera calibration as a geometric inference problem. A printed board gives you known 3D points on a plane. Corner detection gives you the matching 2D pixel locations. Calibration then finds the camera intrinsics, distortion coefficients, and one board pose per image that best explain those correspondences.

You will then evaluate the fit with visual and numerical checks before using the result for planar measurement.

**Prerequisites.** You have already captured calibration images, run `detect_corners.py`, and copied the `data/` and `captured_points/` folders to your laptop.

**How this notebook is structured.** Like the Lab 1a and 1b notebooks, each section has a short teaching cell followed by one or more working code cells and a short reflection prompt. A few exercise cells contain `???` placeholders. Replace them before moving on. Predict what the code will tell you before you run it, then reconcile prediction with observation.


In [11]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml

print('OpenCV :', cv2.__version__)
print('NumPy  :', np.__version__)

if not hasattr(cv2, 'aruco'):
    raise ImportError('cv2.aruco is not available. Install opencv-contrib-python on the laptop.')

cwd = Path.cwd().resolve()
if (cwd / 'board_config.py').exists():
    src_dir = cwd
elif (cwd / 'src' / 'board_config.py').exists():
    src_dir = cwd / 'src'
else:
    raise RuntimeError('Open this notebook from Lab_2/src or Lab_2.')

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from board_config import (
    CAPTURED_POINTS_DIR,
    CHARUCO_MARKER_SIZE_M,
    CHARUCO_SQUARES_X,
    CHARUCO_SQUARES_Y,
    CHECKERBOARD_INNER_CORNERS,
    DATA_DIR,
    SQUARE_SIZE_M,
    charuco_object_points_for_ids,
    create_charuco_board,
)

np.set_printoptions(suppress=True, precision=4)
plt.style.use('default')

npz_path = CAPTURED_POINTS_DIR / 'corners.npz'
yaml_out = CAPTURED_POINTS_DIR / 'intrinsics.yml'

if not npz_path.exists():
    raise FileNotFoundError(f'Could not find {npz_path}. Run detect_corners.py first.')


def resolve_local_image_path(stored_path):
    stored_path = Path(stored_path)
    if stored_path.exists():
        return stored_path

    local_candidate = DATA_DIR / stored_path.name
    if local_candidate.exists():
        return local_candidate

    return local_candidate


print(f'Corners file: {npz_path}')
print(f'Intrinsics output: {yaml_out}')


OpenCV : 4.10.0
NumPy  : 1.23.4
Corners file: /Users/azeez/Google_Drive/Life/PhD/AppCV_2026/AppCV_2026/Lab_2/captured_points/corners.npz
Intrinsics output: /Users/azeez/Google_Drive/Life/PhD/AppCV_2026/AppCV_2026/Lab_2/captured_points/intrinsics.yml


## 1. What data do we already have?

A calibration board is useful because it gives us **correspondences**.

- In the world, we know where the board corners are relative to each other.
- In the image, the detector gives us pixel coordinates for those same corners.
- Across many images with different poses, those 3D to 2D matches constrain the camera model.

The next cell inspects the saved corner file and tells you how many valid frames and point correspondences are available for calibration. It also asks you to identify whether the file came from a checkerboard run or a ChArUco run by looking at the saved array names.


In [ ]:
data = np.load(npz_path, allow_pickle=False)
print('Available arrays in corners.npz:', list(data.files))

if 'objpoints' in data.files and 'imgpoints' in data.files:
    mode = 'checker'
elif 'objpoints' in data.files and 'imgpoints' in data.files:
    mode = 'charuco'
else:
    raise ValueError(f'Unrecognized corners.npz format. Keys were: {data.files}')

image_size = tuple(int(v) for v in data['img_shape'])
valid_files = [resolve_local_image_path(p) for p in data['valid_files'].tolist()]
missing_files = [path for path in valid_files if not path.exists()]

if mode == 'checker':
    objpoints = [frame.astype(np.float32) for frame in data['objpoints']]
    imgpoints = [frame.astype(np.float32) for frame in data['imgpoints']]
    point_counts = [frame.shape[0] for frame in imgpoints]
    expected = CHECKERBOARD_INNER_CORNERS[0] * CHECKERBOARD_INNER_CORNERS[1]
    valid_rule = f'all {expected} checkerboard inner corners were found'
else:
    board = create_charuco_board()
    corners = [frame.astype(np.float32) for frame in data['corners']]
    ids = [frame.astype(np.int32) for frame in data['ids']]
    point_counts = [frame.shape[0] for frame in corners]
    expected = (CHARUCO_SQUARES_X - 1) * (CHARUCO_SQUARES_Y - 1)
    valid_rule = f'the detector kept frames with all {expected} ChArUco corners'

total_points = int(sum(point_counts))

print(f'Calibration mode: {mode}')
print(f'Image size: {image_size[0]} x {image_size[1]}')
print(f'Valid frame count: {len(valid_files)}')
print(f'Points per valid frame: min={min(point_counts)}  max={max(point_counts)}')
print(f'Total 3D-2D correspondences used for calibration: {total_points}')
print(f'Valid-frame rule: {valid_rule}.')

if missing_files:
    print(f'Warning: {len(missing_files)} valid images are not available locally.')
    print(f'Example missing file: {missing_files[0]}')
    print('Copy the matching image set into data/ if you want the overlay and undistortion cells to work.')


Available arrays in corners.npz: ['valid_files', 'objpoints', 'imgpoints', 'img_shape']


ValueError: Unrecognized corners.npz format. Keys were: ['valid_files', 'objpoints', 'imgpoints', 'img_shape']

**Answer inline (one or two sentences each):**

1. Why can the total point count be much larger than the number of valid images?
2. In your own words, what does a **valid frame** mean in this notebook?
3. Why is it useful to capture the same board under many different poses rather than taking many almost-identical images?

*Your answers:*


## 2. Does the board description match the printed board?

Before calibrating, check that the code matches the board you actually printed and measured in class.

If the square size or marker size is wrong, the scale of the calibration and the later measurement step will also be wrong. In the next cell, fill in the board dimensions you measured.

Hint:
- For the checkerboard, use the number of **inner corners**, not the number of black and white squares.
- For the ChArUco board, use the number of **squares** along x and y.


In [ ]:
# Fill these in from the boards you used in class.
# Checkerboard: use INNER corners, not the number of squares.
checker_inner_x = ???
checker_inner_y = ???

# ChArUco: use the number of SQUARES across x and y.
charuco_squares_x = ???
charuco_squares_y = ???

# Measure the printed board in centimetres.
square_size_cm = ???
marker_size_cm = ???

print('Your checkerboard inner corners:', (checker_inner_x, checker_inner_y))
print('Your ChArUco squares:', (charuco_squares_x, charuco_squares_y))
print(f'Your square size: {square_size_cm:.2f} cm')
print(f'Your marker size: {marker_size_cm:.2f} cm')

print('\nReference values from board_config.py')
print('Checkerboard inner corners:', CHECKERBOARD_INNER_CORNERS)
print(f'ChArUco squares: {(CHARUCO_SQUARES_X, CHARUCO_SQUARES_Y)}')
print(f'Square size: {SQUARE_SIZE_M * 100:.2f} cm')
print(f'Marker size: {CHARUCO_MARKER_SIZE_M * 100:.2f} cm')


## 3. Run calibration

Calibration solves for two kinds of quantities.

- **Camera quantities** that stay fixed for this camera and lens. These are the intrinsic matrix `K` and the distortion coefficients `d`.
- **Per-image quantities** that change from frame to frame. These are the rotation and translation of the board relative to the camera.

OpenCV returns a mean reprojection error, but that number alone is not enough. The next sections will inspect the fit more carefully. In the next cell, fill in the OpenCV calibration function names for checkerboard and ChArUco.

Docs hint. Use the official OpenCV docs and search within the page for `calibrateCamera` or `calibrateCameraCharuco`. Read the Python signature there and map it onto the variables in this notebook.
- [OpenCV calib3d docs](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)
- [OpenCV ArUco docs](https://docs.opencv.org/4.x/d9/d6a/group__aruco.html)


In [ ]:
if mode == 'checker':
    ret, K, d, rvecs, tvecs = cv2.???(
        objpoints,
        imgpoints,
        image_size,
        None,
        None,
    )
else:
    ret, K, d, rvecs, tvecs = cv2.aruco.???(
        charucoCorners=corners,
        charucoIds=ids,
        board=board,
        imageSize=image_size,
        cameraMatrix=None,
        distCoeffs=None,
    )

payload = {
    'K': {'data': K.tolist(), 'rows': 3, 'cols': 3},
    'D': {'data': d.ravel().tolist(), 'rows': 1, 'cols': int(d.size)},
}
with open(yaml_out, 'w', encoding='utf-8') as f:
    yaml.safe_dump(payload, f, sort_keys=False)

print(f'Mean reprojection error from OpenCV: {ret:.4f} px')
print('K =\n', K)
print('d =\n', d.ravel())
print(f'Saved intrinsics to {yaml_out}')


**Answer inline (one or two sentences each):**

1. Which quantities above belong to the camera itself, and which belong to one particular image?
2. Where is the intrinsics file being saved on disk?
3. Why is a low mean reprojection error encouraging, but not enough by itself to trust the calibration?

*Your answers:*


## 4. Evaluate the fit across all detected points

The next cell computes a pointwise reprojection error for **every detected corner in every valid image**.

This is why the histogram count can be much larger than the number of images. The histogram is not counting frames. It is counting **corner points**.

The histogram is built from those per-point reprojection errors aggregated across all used images.

The scatter plot uses the distance from the **principal point** `(cx, cy)` measured in pixels. This is an image-plane radius, not a 3D distance from the physical camera. The cell also asks you to fill in the OpenCV call that projects known 3D board points back into the image.

Docs hint. Use the same calib3d page and search for `projectPoints`. The docs show the Python function name and the argument order.
- [OpenCV calib3d docs](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)


In [ ]:
# Fill in the OpenCV function name directly in the call below.

frame_reports = []
all_errors = []
all_radial = []
all_xy = []

for idx, file_path in enumerate(valid_files):
    if mode == 'checker':
        objp = objpoints[idx]
        detected = imgpoints[idx].reshape(-1, 2)
    else:
        objp = charuco_object_points_for_ids(board, ids[idx])
        detected = corners[idx].reshape(-1, 2)

    projected, _ = cv2.???(objp, rvecs[idx], tvecs[idx], K, d)
    projected = projected.reshape(-1, 2)

    point_errors = np.linalg.norm(projected - detected, axis=1)
    radii = np.linalg.norm(detected - K[:2, 2], axis=1)

    all_errors.extend(point_errors.tolist())
    all_radial.extend(radii.tolist())
    all_xy.append(detected)

    frame_reports.append(
        {
            'index': idx,
            'file': file_path.name,
            'mean_error': float(point_errors.mean()),
            'max_error': float(point_errors.max()),
            'points': int(len(point_errors)),
        }
    )

all_errors = np.asarray(all_errors, dtype=np.float32)
all_radial = np.asarray(all_radial, dtype=np.float32)
all_xy = np.vstack(all_xy)

print(f'Total detected points used in calibration: {len(all_errors)}')
print(f'Mean point error: {all_errors.mean():.4f} px')
print(f'Median point error: {np.median(all_errors):.4f} px')
print(f'95th percentile point error: {np.percentile(all_errors, 95):.4f} px')

worst_frames = sorted(frame_reports, key=lambda item: item['mean_error'], reverse=True)[:5]
print('\nWorst frames by mean reprojection error:')
for item in worst_frames:
    print(
        f"{item['index']:>2}  {item['file']:<20}  mean={item['mean_error']:.4f}px  max={item['max_error']:.4f}px  points={item['points']}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

upper = max(np.percentile(all_errors, 99), 0.5)
axes[0].hist(all_errors, bins=30, range=(0, upper), color='0.25')
axes[0].set_title('Reprojection error distribution')
axes[0].set_xlabel('Error in pixels')
axes[0].set_ylabel('Count')

axes[1].scatter(all_radial, all_errors, s=8, alpha=0.35, color='tab:blue')
axes[1].set_title('Error versus distance from the principal point')
axes[1].set_xlabel('Radius in pixels')
axes[1].set_ylabel('Error in pixels')

plt.tight_layout()
plt.show()


**Observations.**

1. Does the total point count match what you would expect from `valid frames x corners per frame`?
2. Do the errors seem tightly concentrated, or is there a long tail of worse points?
3. Does residual error clearly grow with distance from the principal point, or is the trend weak?
4. If a few frames are much worse than the rest, what might that say about blur, glare, or poor detections?

*Your observations:*


## 5. Inspect one frame directly

A low average error is not enough on its own. The most direct visual check is to compare:

- **green points** for the detected corners
- **red points** for the corners reprojected by the calibrated model

The same cell also shows an undistorted version of the example image. In the next cell, fill in the OpenCV function that undistorts the image using `K` and `d`.

Docs hint. Use the same calib3d page and search for `undistort`. Read the Python signature and match it to `example_bgr`, `K`, and `d`.
- [OpenCV calib3d docs](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)


In [ ]:
example_idx = len(valid_files) // 2
example_path = valid_files[example_idx]
example_bgr = cv2.imread(str(example_path))
if example_bgr is None:
    raise FileNotFoundError(
        f'Could not read {example_path}. If corners.npz came from the Pi, copy the matching images into data/ and rerun the notebook cells.'
    )

if mode == 'checker':
    example_objp = objpoints[example_idx]
    example_detected = imgpoints[example_idx].reshape(-1, 2)
else:
    example_objp = charuco_object_points_for_ids(board, ids[example_idx])
    example_detected = corners[example_idx].reshape(-1, 2)

example_projected, _ = cv2.projectPoints(
    example_objp,
    rvecs[example_idx],
    tvecs[example_idx],
    K,
    d,
)
example_projected = example_projected.reshape(-1, 2)

overlay = example_bgr.copy()
for x, y in example_detected:
    cv2.circle(overlay, (int(round(x)), int(round(y))), 5, (0, 255, 0), 1)
for u, v in example_projected:
    cv2.circle(overlay, (int(round(u)), int(round(v))), 3, (0, 0, 255), -1)

undistorted = cv2.???(example_bgr, K, d)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Green detected, red reprojected\n{example_path.name}')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
axes[1].set_title('Undistorted image')
axes[1].axis('off')

plt.tight_layout()
plt.show()


**Answer inline.** If the red and green points visibly separate in some region, what does that tell you that a single average error number might hide?

*Your answer:*


## 6. Where does the model struggle in the image?

The next cell adds two more spatial diagnostics.

- The **vector field** shows how undistortion moves image points across the frame.
- The **heatmap** shows where reprojection error tends to be larger or smaller.

Together, these help you answer whether the model struggles mainly near the borders, mainly in one corner, or fairly uniformly.


In [ ]:
grid_x = np.linspace(0, image_size[0] - 1, 18)
grid_y = np.linspace(0, image_size[1] - 1, 12)
gx, gy = np.meshgrid(grid_x, grid_y)
grid_points = np.stack([gx, gy], axis=-1).astype(np.float32).reshape(-1, 1, 2)

undistorted_grid = cv2.undistortPoints(grid_points, K, d, P=K).reshape(-1, 2)
original_grid = grid_points.reshape(-1, 2)
shift = undistorted_grid - original_grid

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].quiver(
    original_grid[:, 0],
    original_grid[:, 1],
    shift[:, 0],
    shift[:, 1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='tab:red',
)
axes[0].set_xlim(0, image_size[0])
axes[0].set_ylim(image_size[1], 0)
axes[0].set_aspect('equal')
axes[0].set_title('Undistortion vector field')
axes[0].set_xlabel('u')
axes[0].set_ylabel('v')

hb = axes[1].hexbin(
    all_xy[:, 0],
    all_xy[:, 1],
    C=all_errors,
    reduce_C_function=np.mean,
    gridsize=18,
    cmap='magma',
)
axes[1].set_xlim(0, image_size[0])
axes[1].set_ylim(image_size[1], 0)
axes[1].set_title('Mean error by image location')
axes[1].set_xlabel('u')
axes[1].set_ylabel('v')
cb = fig.colorbar(hb, ax=axes[1])
cb.set_label('Mean error in pixels')

plt.tight_layout()
plt.show()


**Reflection.**

1. Does undistortion mainly pull points inward, push them outward, or only move them slightly?
2. Does the heatmap suggest that one part of the image is less trustworthy than the rest?
3. If you were to recapture data, which region of the image would you try to constrain better?

*Your reflection:*


## 7. Explore the intrinsic and distortion parameters

This interactive block is provided for exploration rather than code writing.

Start with the **Intrinsics** tab. Change `fx`, `fy`, `cx`, and `cy` and watch how the red reprojected corners move relative to the green detected corners.

Then switch to the **Distortion** tab. Perturb the radial (`k1`, `k2`, `k3`) and tangential (`p1`, `p2`) coefficients and see which parts of the image move most strongly.

As you do this, ask yourself which board positions in the image are most informative for judging radial distortion in the first place.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as exc:
    raise ImportError('ipywidgets is required for the interactive section. Install it with pip install ipywidgets.') from exc

base_K = K.copy().astype(np.float64)
base_d = np.ravel(d).astype(np.float64)
if base_d.size < 5:
    base_d = np.pad(base_d, (0, 5 - base_d.size))


def make_slider(label, value, coarse_factor=3.0, fine_step=1e-3, hard_max=5.0):
    span = max(abs(float(value)) * coarse_factor, 0.05)
    span = min(span, hard_max)
    return widgets.FloatSlider(
        value=float(value),
        min=float(value) - span,
        max=float(value) + span,
        step=fine_step,
        description=label,
        style={'description_width': 'initial'},
        continuous_update=False,
    )


fx_slider = widgets.FloatSlider(
    value=float(base_K[0, 0]),
    min=float(base_K[0, 0] * 0.8),
    max=float(base_K[0, 0] * 1.2),
    step=1.0,
    description='fx',
    style={'description_width': 'initial'},
    continuous_update=False,
)
fy_slider = widgets.FloatSlider(
    value=float(base_K[1, 1]),
    min=float(base_K[1, 1] * 0.8),
    max=float(base_K[1, 1] * 1.2),
    step=1.0,
    description='fy',
    style={'description_width': 'initial'},
    continuous_update=False,
)
cx_slider = widgets.FloatSlider(
    value=float(base_K[0, 2]),
    min=float(base_K[0, 2] - 50.0),
    max=float(base_K[0, 2] + 50.0),
    step=0.5,
    description='cx',
    style={'description_width': 'initial'},
    continuous_update=False,
)
cy_slider = widgets.FloatSlider(
    value=float(base_K[1, 2]),
    min=float(base_K[1, 2] - 50.0),
    max=float(base_K[1, 2] + 50.0),
    step=0.5,
    description='cy',
    style={'description_width': 'initial'},
    continuous_update=False,
)

k1_slider = make_slider('k1', base_d[0])
k2_slider = make_slider('k2', base_d[1])
p1_slider = make_slider('p1', base_d[2], coarse_factor=3.0, fine_step=1e-4, hard_max=0.2)
p2_slider = make_slider('p2', base_d[3], coarse_factor=3.0, fine_step=1e-4, hard_max=0.2)
k3_slider = make_slider('k3', base_d[4], coarse_factor=1.5, fine_step=1e-2, hard_max=10.0)

reset_intr_btn = widgets.Button(
    description='Reset intrinsics',
    tooltip='Restore the calibrated intrinsic values',
)
reset_dist_btn = widgets.Button(
    description='Reset distortion',
    tooltip='Restore the calibrated distortion values',
)


def reset_intr_sliders(_):
    fx_slider.value = float(base_K[0, 0])
    fy_slider.value = float(base_K[1, 1])
    cx_slider.value = float(base_K[0, 2])
    cy_slider.value = float(base_K[1, 2])


def reset_dist_sliders(_):
    for slider, value in zip(
        [k1_slider, k2_slider, p1_slider, p2_slider, k3_slider],
        base_d[:5],
    ):
        slider.value = float(value)


reset_intr_btn.on_click(reset_intr_sliders)
reset_dist_btn.on_click(reset_dist_sliders)


def draw_projection_overlay(current_K, current_d, title):
    proj_pts, _ = cv2.projectPoints(
        example_objp,
        rvecs[example_idx],
        tvecs[example_idx],
        current_K,
        current_d.reshape(-1, 1),
    )
    proj_pts = proj_pts.reshape(-1, 2)

    vis = example_bgr.copy()
    for x, y in example_detected:
        cv2.circle(vis, (int(round(x)), int(round(y))), 5, (0, 255, 0), 1)
    for u, v in proj_pts:
        cv2.circle(vis, (int(round(u)), int(round(v))), 3, (0, 0, 255), -1)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
    plt.show()
    plt.close(fig)


intr_controls = widgets.VBox([
    widgets.HBox([reset_intr_btn]),
    fx_slider,
    fy_slider,
    cx_slider,
    cy_slider,
])

dist_controls = widgets.VBox([
    widgets.HBox([reset_dist_btn]),
    k1_slider,
    k2_slider,
    p1_slider,
    p2_slider,
    k3_slider,
])

tab = widgets.Tab(children=[intr_controls, dist_controls])
tab.set_title(0, 'Intrinsics')
tab.set_title(1, 'Distortion')

output = widgets.Output()


def render_current(*_):
    output.clear_output(wait=True)
    with output:
        if tab.selected_index == 0:
            trial_K = base_K.copy()
            trial_K[0, 0] = fx_slider.value
            trial_K[1, 1] = fy_slider.value
            trial_K[0, 2] = cx_slider.value
            trial_K[1, 2] = cy_slider.value
            draw_projection_overlay(
                trial_K,
                base_d,
                'Green detected, red reprojected after intrinsic changes',
            )
        else:
            trial_d = base_d.copy()
            trial_d[:5] = [
                k1_slider.value,
                k2_slider.value,
                p1_slider.value,
                p2_slider.value,
                k3_slider.value,
            ]
            draw_projection_overlay(
                base_K,
                trial_d,
                'Green detected, red reprojected after distortion changes',
            )


for slider in [fx_slider, fy_slider, cx_slider, cy_slider, k1_slider, k2_slider, p1_slider, p2_slider, k3_slider]:
    slider.observe(render_current, names='value')

tab.observe(render_current, names='selected_index')

display(widgets.VBox([tab, output]))
render_current()


## 8. Distortion observations

Use the reference image below to help describe the type of distortion you think you are seeing in your own calibration result.

![Distortion reference](imgs/distortion.png)

**Write a short observation for yourself or for your lab notes:**

1. Does your camera look closer to barrel distortion, pincushion distortion, or only a small effect?
2. Do the biggest changes happen near the image borders or near the principal point?
3. Does the effect look mostly radial, or do you also see some tangential skew?

*Your observations:*


## 9. Next step - planar measurement

The saved `intrinsics.yml` can now be used with `measure_object.py`.

That measurement step only works when the clicked object points lie on the same physical plane as the calibration board. In other words, the method uses the board plane constraint `Z = 0` to turn image points back into real-world distances on that plane.

If you have time, repeat the workflow with the other board type and compare:

- the number of valid frames
- the reprojection-error plots
- the spatial diagnostics
- the clarity of the final measurement step
